# 01. Exploratory Data Analysis (EDA)

1. Dataset Structure Inspection
2. Data Integration
3. Data Quality Assessment
   - 3.1 Dataset Overview
   - 3.2 Missing Song ID Investigation
   - 3.3 Empty String Analysis
   - 3.4 Missing Value Overlap Analysis
   - 3.5 Duplicate Analysis
   - 3.6 Same-Month Duplicate Investigation
   - 3.7 Lyrics Availability Analysis
4. Dataset Source & Reliability Assessment
   - 4.1 Dataset Source Inspection
   - 4.2 Stratified Random Sampling
   - 4.3 Release Date Consistency Check
   - 4.4 Temporal Discrepancy Distribution
   - 4.5 Flagged Record Sample
   - 4.6 Export Temporal Discrepancy Records
   - 4.7 Monthly Record Counts
   - 4.8 Manual Verification Checklist
   - 4.9 Month-Level Anomaly Detection
5. EDA Summary

## 1. Dataset Structure Inspection

### Objective
Examine the structure and contents of the raw Melon Monthly Chart dataset (2000–2023).

### Procedure
- Count the total number of JSON files.
- Inspect a sample JSON file.
- Identify available song metadata, chart information, and lyrics.

### Results
- Total JSON files: 25,876
- Data format: JSON
- Lyrics format: List of strings
- Available information: Song metadata, chart year, month, rank, and lyrics

### Key Findings
- The dataset contains structured song metadata and lyrics suitable for text mining and NLP analysis.
- Chart dates and release dates are stored separately, allowing temporal analysis based on different time references.

In [1]:
from pathlib import Path
import json

DATA_DIR = Path("../data/raw/melon")

json_files = sorted(DATA_DIR.rglob("*.json"))

print(f"Total JSON files: {len(json_files):,}")

with open(json_files[0], "r", encoding="utf-8") as f:
    sample = json.load(f)

print(json.dumps(sample, ensure_ascii=False, indent=2)[:4000])

Total JSON files: 25,876
{
  "info": [
    {
      "year": 2000,
      "month": 1,
      "rank": 1,
      "type": "월별차트",
      "website": "Melon"
    },
    ""
  ],
  "song_id": "38541",
  "song_name": "부메랑",
  "album": "Growing",
  "release_date": "1999.11.20",
  "artist": "김건모",
  "genre": "발라드",
  "lyric_writer": "문창배",
  "composer": "문창배",
  "arranger": "문창배",
  "lyrics": {
    "row_num": 42,
    "lines": [
      "다가오지마 더 이상",
      "위로될 수 없다는 것을",
      "모르는 것처럼",
      "걱정하지마 차라리",
      "편하기까지 하다니까",
      "믿고 싶지 않니",
      "나를 걱정해서 내게 오는 거라면",
      "이제는 아무렇지도 않아",
      "니가 후회해서 돌아오는 거라도",
      "이번에는 내가 싫어",
      "다가오지마 더 이상",
      "위로될 수 없다는 것을",
      "모르는 것처럼",
      "걱정하지마 차라리",
      "편하기까지 하다니까",
      "믿고 싶지 않니",
      "이제야 간신히 너의 그늘 속에서",
      "벗어나 담담할 수 있는데",
      "난 항상 너에게 목을 메고 있다고",
      "그렇게 착각 한 거니",
      "다가오지마 더 이상",
      "위로될 수 없다는 것을",
      "모르는 것처럼",
      "걱정하지마 차라리",
      "편하기까지 하다니까",
      "믿고 싶지 않니",
      "다가오지마",
      "다가오지마 더 이상",
      "


## 2. Data Integration

### Objective
Consolidate individual JSON files into a structured DataFrame for subsequent analysis.

### Procedure
- Extract song metadata, chart information, and lyrics.
- Combine all JSON records into a single pandas DataFrame.
- Check for errors during data loading.

### Results
- Total records: 25,876
- Total columns: 11
- Data loading errors: 0

### Key Findings
- All JSON files were successfully integrated into a single DataFrame.
- The integrated dataset provides the necessary information for data quality assessment and preprocessing.


In [2]:
import pandas as pd

records = []
errors = []

for file in json_files:
    try:
        with open(file, "r", encoding="utf-8") as f:
            data = json.load(f)

        info = data.get("info") or []
        chart_info = info[0] if info and isinstance(info[0], dict) else {}

        lyrics = data.get("lyrics") or {}

        records.append({
            "song_id": data.get("song_id"),
            "song_name": data.get("song_name"),
            "artist": data.get("artist"),
            "album": data.get("album"),
            "release_date": data.get("release_date"),
            "genre": data.get("genre"),
            "chart_year": chart_info.get("year"),
            "chart_month": chart_info.get("month"),
            "chart_rank": chart_info.get("rank"),
            "lyrics": lyrics.get("lines") if isinstance(lyrics, dict) else None,
            "source_file": str(file.relative_to(DATA_DIR))
        })

    except Exception as e:
        errors.append((str(file), str(e)))

df = pd.DataFrame(records)

print(f"Total records: {len(df):,}")
print(f"Read errors: {len(errors):,}")

display(df.head())

Total records: 25,876
Read errors: 0


,song_id,song_name,artist,album,release_date,genre,chart_year,chart_month,chart_rank,lyrics,source_file
0,38541,부메랑,김건모,Growing,1999.11.20,발라드,2000,1,1,"[다가오지마 더 이상, 위로될 수 없다는 것을, 모르는 것처럼, 걱정하지마 차라리,...",monthly-chart/melon-2000/melon-2000-01/melon-m...
1,85257,바꿔,이정현,Let's Go To My Star,1999.10,댄스,2000,1,2,"[모두 제정신이 아니야 다들, 미쳐가고만 있어, 어느 누굴 믿어 어찌믿어, 더는 못...",monthly-chart/melon-2000/melon-2000-01/melon-m...
2,91324,사랑해 그리고 기억해,god,Chapter 2,1999.11.25,발라드,2000,1,3,"[Uh 사랑하는 사람을, 보내야만 했던, 모든 사람들에게 바칩니다, 돌아보면 너무나...",monthly-chart/melon-2000/melon-2000-01/melon-m...
3,85674,영원,스카이,Final Fantasy,1999.11.16,"발라드, 록/메탈",2000,1,4,"[너와의 행복했었던, 또 아름다웠던 추억, 우연히 지나친, 그런 시간만은 아닐 거야...",monthly-chart/melon-2000/melon-2000-01/melon-m...
4,91320,실연,코요태,실연,1999.11.06,댄스,2000,1,5,"[너 떠난후, 많이 울었어, 혼자가, 되어버렸어, 나 다시는 널 볼 수, 없게 된거...",monthly-chart/melon-2000/melon-2000-01/melon-m...


## 3. Data Quality Assessment

### Objective
Evaluate the completeness, consistency, and uniqueness of the integrated dataset before preprocessing.

### Procedure
- Examine data types and missing values.
- Identify duplicate song IDs and chart entries.
- Calculate the number of unique songs.
- Assess the availability and completeness of lyrics.

### Expected Output
- Missing value summary
- Duplicate record statistics
- Unique song count
- Lyrics availability statistics

### 3.1 Dataset Overview

#### Objective
Assess the overall structure, data types, and missing values of the integrated dataset.

#### Procedure
- Examine dataset dimensions and column data types.
- Identify missing values using standard null-value detection.
- Calculate missing-value counts and percentages.

#### Results
- Dataset dimensions: 25,876 rows × 11 columns
- Missing song IDs: 346 (1.34%)
- Missing values in other columns: 0 (based on null-value detection)

#### Key Findings
- Song ID is the only column containing null values.
- Standard null-value detection does not identify empty strings, requiring further investigation.

In [3]:
print("Dataset shape:", df.shape)
print("\nColumn data types:")
print(df.dtypes)

print("\nMissing values:")
display(
    pd.DataFrame({
        "Missing Count": df.isna().sum(),
        "Missing (%)": (df.isna().mean() * 100).round(2)
    })
)

Dataset shape: (25876, 11)

Column data types:
song_id            str
song_name          str
artist             str
album              str
release_date       str
genre              str
chart_year       int64
chart_month      int64
chart_rank       int64
lyrics          object
source_file        str
dtype: object

Missing values:


,Missing Count,Missing (%)
song_id,346,1.34
song_name,0,0.00
artist,0,0.00
album,0,0.00
release_date,0,0.00
genre,0,0.00
chart_year,0,0.00
chart_month,0,0.00
chart_rank,0,0.00
lyrics,0,0.00


### 3.2 Missing Song ID Investigation

#### Objective
Investigate the distribution and characteristics of records with missing song IDs.

#### Procedure
- Identify records with missing song IDs.
- Examine their distribution across chart years.
- Inspect sample records to identify recurring patterns.

#### Results
- Missing song IDs: 346 records (1.34%)
- Affected chart years: 2000–2017
- Several songs with missing IDs appear across multiple monthly charts.

#### Key Findings
- Missing song IDs are concentrated in earlier years.
- Some missing-ID records represent repeated chart appearances of the same songs.
- Missing-ID records should be retained until song identification and duplicate analysis are completed.
- Sample records also reveal empty release dates that were not detected by standard null-value checks.

In [4]:
missing_id = df[df["song_id"].isna()].copy()

print(f"Total missing song IDs: {len(missing_id):,}")

print("\nMissing song IDs by year:")
display(
    missing_id.groupby("chart_year")
    .size()
    .to_frame("missing_count")
)

print("\nSample records with missing song IDs:")
display(
    missing_id[
        ["song_name", "artist", "release_date",
         "chart_year", "chart_month", "chart_rank"]
    ].head(15)
)

Total missing song IDs: 346

Missing song IDs by year:


,missing_count
chart_year,
2000,41
2001,34
2002,10
2003,11
2004,24
2005,35
2006,30
2007,8
2008,10



Sample records with missing song IDs:


,song_name,artist,release_date,chart_year,chart_month,chart_rank
15,겨울이야기,DJ DOC,,2000,1,16
26,Say You Will,유승준,,2000,1,27
71,상처 (Special Re-Mix),조성모,,2000,1,50
110,약속,유영석,,2000,2,36
128,겨울이야기,DJ DOC,,2000,2,50
132,Say You Will,유승준,,2000,2,50
159,약속,유영석,,2000,3,23
171,Love,윤현석,,2000,3,35
192,마린 일병 구하기(Remix),허니 패밀리,,2000,3,50
218,Love,윤현석,,2000,4,19


### 3.3 Empty String Analysis

#### Objective
Identify missing metadata represented by empty strings or whitespace rather than null values.

#### Procedure
- Examine text-based metadata columns.
- Detect null values, empty strings, and whitespace-only entries.
- Calculate the total number and percentage of missing values for each column.


#### Results
- Total records: 25,876
- Missing song IDs: 346 (1.34%)
- Missing release dates: 348 (1.34%)
- Missing genres: 348 (1.34%)
- No missing values were detected in song names, artists, or albums.

#### Key Findings
- Standard null-value detection underestimated missing values because some metadata fields contain empty strings.
- Missing values are limited to three metadata columns: `song_id`, `release_date`, and `genre`.
- The missing-value rates are relatively low (approximately 1.34%).
- Further investigation is needed to determine whether missing values occur in the same records.

In [5]:
text_columns = [
    "song_id", "song_name", "artist", "album",
    "release_date", "genre"
]

missing_summary = pd.DataFrame({
    "Missing Count": [
        (df[col].isna() | df[col].str.strip().eq("")).sum()
        for col in text_columns
    ]
}, index=text_columns)

missing_summary["Missing (%)"] = (
    missing_summary["Missing Count"] / len(df) * 100
).round(2)

display(missing_summary)

,Missing Count,Missing (%)
song_id,346,1.34
song_name,0,0.00
artist,0,0.00
album,0,0.00
release_date,348,1.34
genre,348,1.34


### 3.4 Missing Value Overlap Analysis

#### Objective
Examine whether missing values in song IDs, release dates, and genres occur within the same records.

#### Procedure
- Create missing-value indicators for the three metadata fields.
- Identify combinations of missing values.
- Calculate the number of records affected by each missing-value pattern.

#### Results
- Complete records: 25,528 (98.66%)
- Records missing all three fields: 346 (1.34%)
- Records missing only release date and genre: 2 (0.01%)
- Total records with at least one missing field: 348 (1.34%)

#### Key Findings
- Missing values are highly concentrated in a small subset of records.
- All 346 records with missing song IDs also lack release dates and genres.
- Only two additional records have missing release dates and genres while retaining valid song IDs.
- These records should be investigated before deciding whether to exclude or recover missing metadata.

In [6]:
columns = ["song_id", "release_date", "genre"]

missing_flags = pd.DataFrame({
    col: df[col].isna() | df[col].str.strip().eq("")
    for col in columns
})

print("Missing value combinations:")
display(
    missing_flags.value_counts()
    .reset_index(name="record_count")
    .sort_values("record_count", ascending=False)
)

print("\nRecords with at least one missing field:")
print(f"{missing_flags.any(axis=1).sum():,}")

print("\nRecords with all three fields missing:")
print(f"{missing_flags.all(axis=1).sum():,}")


Missing value combinations:


,song_id,release_date,genre,record_count
0,False,False,False,25528
1,True,True,True,346
2,False,True,True,2



Records with at least one missing field:
348

Records with all three fields missing:
346


### 3.5 Duplicate Analysis

#### Objective
Identify repeated song records and evaluate their implications for the unit of analysis.

#### Procedure
- Count unique song IDs among records with valid identifiers.
- Identify repeated song appearances across monthly charts.
- Check for duplicate entries within the same chart month.
- Examine songs with the highest number of chart appearances.

#### Results
- Total chart records: 25,876
- Records with valid song IDs: 25,530
- Unique song IDs: 8,115
- Repeated song ID records: 17,415 (68.2% of valid-ID records)
- Duplicate entries within the same month: 6
- Most frequently appearing song: "Spring Day" by BTS (80 appearances)

#### Key Findings
- A substantial proportion of records represent repeated monthly chart appearances rather than unique songs.
- Analyzing all chart records independently would give greater weight to songs that remained on the charts longer.
- Song-level data is more appropriate for the primary linguistic analysis, while chart-level data should be preserved for supplementary analysis.
- Records with missing song IDs and duplicate entries within the same month require further investigation before deduplication.

In [7]:
# Exclude missing IDs when counting unique song IDs
valid_id = df[
    df["song_id"].notna() &
    df["song_id"].str.strip().ne("")
]

print(f"Total chart records: {len(df):,}")
print(f"Records with valid song IDs: {len(valid_id):,}")
print(f"Unique song IDs: {valid_id['song_id'].nunique():,}")

print(
    f"Repeated song ID records: "
    f"{valid_id['song_id'].duplicated().sum():,}"
)

# Check duplicates within the same monthly chart
monthly_duplicates = valid_id.duplicated(
    subset=["song_id", "chart_year", "chart_month"]
)

print(f"Duplicate entries within the same month: {monthly_duplicates.sum():,}")

# Most frequently appearing songs
top_repeated = (
    valid_id.groupby(["song_id", "song_name", "artist"])
    .size()
    .reset_index(name="chart_appearances")
    .sort_values("chart_appearances", ascending=False)
    .head(10)
)

display(top_repeated)


Total chart records: 25,876
Records with valid song IDs: 25,530
Unique song IDs: 8,115
Repeated song ID records: 17,415
Duplicate entries within the same month: 6


,song_id,song_name,artist,chart_appearances
2635,30244931,봄날,방탄소년단,80
3050,30962526,"모든 날, 모든 순간 (Every day, Every Moment)",폴킴,66
3752,32508053,이제 나만 믿어요,임영웅,42
3590,32061975,"어떻게 이별까지 사랑하겠어, 널 사랑하는 거지",AKMU (악뮤),41
3460,31737197,작은 것들을 위한 시 (Boy With Luv) (Feat. Halsey),방탄소년단,39
3845,32872978,Dynamite,방탄소년단,38
3505,31853557,오늘도 빛나는 너에게 (To You My Light) (Feat.이라온),마크툽 (MAKTUB),36
2674,30314784,밤편지,아이유,35
3252,31388145,너를 만나,폴킴,35
3648,32183386,Blueming,아이유,35


### 3.6 Same-Month Duplicate Investigation

#### Objective
Investigate duplicate song entries within the same monthly chart.

#### Procedure
- Identify records sharing the same song ID, chart year, and chart month.
- Compare song metadata and chart rankings.
- Examine the distribution of duplicate entries across years.

#### Results
- Duplicate groups: 6
- Total records involved: 12
- Affected years: 2000–2003
- Three groups contain identical chart ranks, while three groups have different ranks.

#### Key Findings
- A small number of songs appear more than once within the same monthly chart.
- Some duplicates share identical rankings, while others have conflicting rankings.
- The cause of these duplicate entries cannot be determined from the current results alone.
- Duplicate records should be reviewed before constructing the final chart-level dataset.

In [8]:
duplicate_records = valid_id[
    valid_id.duplicated(
        subset=["song_id", "chart_year", "chart_month"],
        keep=False
    )
].sort_values(
    ["chart_year", "chart_month", "song_id", "chart_rank"]
)

display(
    duplicate_records[
        [
            "song_id", "song_name", "artist",
            "chart_year", "chart_month",
            "chart_rank", "source_file"
        ]
    ]
)

print(f"Total records involved: {len(duplicate_records):,}")
print(
    "Duplicate groups:",
    duplicate_records.groupby(
        ["song_id", "chart_year", "chart_month"]
    ).ngroups
)


,song_id,song_name,artist,chart_year,chart_month,chart_rank,source_file
248,494157,다 알아요,양파,2000,4,49,monthly-chart/melon-2000/melon-2000-04/melon-m...
263,494157,다 알아요,양파,2000,4,50,monthly-chart/melon-2000/melon-2000-04/melon-m...
325,85257,바꿔,이정현,2000,5,50,monthly-chart/melon-2000/melon-2000-05/melon-m...
333,85257,바꿔,이정현,2000,5,50,monthly-chart/melon-2000/melon-2000-05/melon-m...
471,4077,오! 가니,컨츄리 꼬꼬,2000,8,18,monthly-chart/melon-2000/melon-2000-08/melon-m...
494,4077,오! 가니,컨츄리 꼬꼬,2000,8,41,monthly-chart/melon-2000/melon-2000-08/melon-m...
893,84375,꿈꾸는 소녀,YB (윤도현 밴드),2001,2,50,monthly-chart/melon-2001/melon-2001-02/melon-m...
898,84375,꿈꾸는 소녀,YB (윤도현 밴드),2001,2,50,monthly-chart/melon-2001/melon-2001-02/melon-m...
1058,8921,오빠,왁스,2001,5,34,monthly-chart/melon-2001/melon-2001-05/melon-m...
1059,8921,오빠,왁스,2001,5,35,monthly-chart/melon-2001/melon-2001-05/melon-m...


Total records involved: 12
Duplicate groups: 6


### 3.7 Lyrics Availability Analysis

#### Objective
Assess the availability and completeness of lyrics in the dataset to determine their suitability for NLP analysis.

#### Procedure
- Identify records containing valid, non-empty lyrics.
- Calculate the number and percentage of records with and without valid lyrics.
- Examine the distribution of lyric line counts.
- Identify records with missing lyrics across chart years.

#### Results
- Total records: 25,876
- Records with valid lyrics: 25,236 (97.53%)
- Records without valid lyrics: 640 (2.47%)
- Average lyric line count: 51.76
- Median lyric line count: 47
- Standard deviation: 24.12
- Minimum lyric line count: 0
- Maximum lyric line count: 201
- Interquartile range (IQR): 35–66 lines

#### Key Findings
- Most records (97.53%) contain non-empty lyrics, indicating high lyrics availability.
- A total of 640 records lack usable lyrics and require further investigation before NLP analysis.
- Lyric lengths vary considerably across records, which may affect length-sensitive linguistic metrics such as lexical diversity.
- Lyrics availability does not guarantee textual accuracy or completeness, highlighting the need for additional dataset reliability validation.

In [9]:
def has_valid_lyrics(lines):
    return (
        isinstance(lines, list)
        and any(
            isinstance(line, str) and line.strip()
            for line in lines
        )
    )

df["has_lyrics"] = df["lyrics"].apply(has_valid_lyrics)

df["lyric_line_count"] = df["lyrics"].apply(
    lambda x: sum(
        1 for line in x
        if isinstance(line, str) and line.strip()
    ) if isinstance(x, list) else 0
)

print(f"Total records: {len(df):,}")
print(f"Records with valid lyrics: {df['has_lyrics'].sum():,}")
print(f"Records without valid lyrics: {(~df['has_lyrics']).sum():,}")
print(f"Lyrics availability: {df['has_lyrics'].mean():.2%}")

print("\nLyric line count statistics:")
display(df["lyric_line_count"].describe())

print("\nRecords without valid lyrics by year:")
display(
    df.loc[~df["has_lyrics"]]
      .groupby("chart_year")
      .size()
      .to_frame("missing_lyrics")
)


Total records: 25,876
Records with valid lyrics: 25,236
Records without valid lyrics: 640
Lyrics availability: 97.53%

Lyric line count statistics:


count    25876.000000
mean        51.763178
std         24.116778
min          0.000000
25%         35.000000
50%         47.000000
75%         66.000000
max        201.000000
Name: lyric_line_count, dtype: float64


Records without valid lyrics by year:


,missing_lyrics
chart_year,
2000,48
2001,38
2002,16
2003,17
2004,30
2005,35
2006,37
2007,26
2008,16


## 4. Dataset Source & Reliability Assessment

### Objective
Assess whether the dataset accurately reflects the official Melon Monthly Chart.

### 4.1 Dataset Source Inspection

#### Objective
Evaluate the credibility of the dataset source by reviewing its documentation, coverage, and collection methodology.

#### Procedure
- Review the original GitHub repository and README.
- Identify the stated data source, coverage, and structure.
- Examine the documentation of data collection and validation procedures.
- Compare the documented dataset size with the actual number of JSON files.

#### Results
- Repository: EX3exp/Kpop-lyric-datasets
- Original data source: Melon Monthly Top 100
- Coverage: January 2000 – October 2023
- Data format: JSON
- Documented dataset size: 25,696 songs
- Actual JSON files: 25,876
- Detailed collection and validation procedures are not provided in the README.

#### Key Findings
- The dataset provides structured song metadata and lyrics attributed to Melon.
- The discrepancy between the documented song count and actual JSON file count requires clarification.
- The README does not provide sufficient methodological details to independently establish data accuracy.
- External validation against official Melon records is necessary before using the dataset for research.

In [10]:
# 4.1 Dataset Source Inspection

from pathlib import Path

DATA_DIR = Path("../data/raw/melon")

# Locate dataset documentation
readme_files = list(DATA_DIR.rglob("README*"))

print(f"README files found: {len(readme_files)}")

for file in readme_files:
    print(f"\nFile: {file}")
    print("-" * 60)

    content = file.read_text(encoding="utf-8")
    print(content[:10000])


README files found: 0


### 4.2 Stratified Random Sampling

#### Objective
Draw a year-stratified sample of songs for reliability checks.

#### Procedure
- Keep records with valid song IDs and lyrics.
- Keep one record per song per year.
- Sample 5 records per chart year (`random_state=42`).

#### Results
- Total validation samples: 120 (5 × 24 years)

In [11]:

# Select records with valid song IDs and lyrics
validation_pool = df[
    df["song_id"].notna() &
    df["has_lyrics"]
].copy()

# Keep one chart record per song per year
validation_pool = validation_pool.drop_duplicates(
    subset=["chart_year", "song_id"]
)

# Sample 5 records from each year
validation_sample = (
    validation_pool.groupby("chart_year", group_keys=False)
    .sample(n=5, random_state=42)
    .sort_values(["chart_year", "song_name"])
    .reset_index(drop=True)
)

print(f"Total validation samples: {len(validation_sample)}")

display(
    validation_sample[
        ["song_id", "song_name", "artist",
         "release_date", "chart_year",
         "chart_month", "chart_rank"]
    ]
)


Total validation samples: 120


,song_id,song_name,artist,release_date,chart_year,chart_month,chart_rank
0,1564057,I Believe,이수영,1999.11.17,2000,1,7
1,4078,Kiss,컨츄리 꼬꼬,2000.06.12,2000,8,33
2,85678,반전(反轉),스카이,1999.11.16,2000,3,20
3,489869,연가,유승준,2001.09.01,2000,1,36
4,104507,태어나서 처음,DJ Wreckx (디제이렉스),2000.06.05,2000,6,50
...,...,...,...,...,...,...,...
115,36713849,Love Lee,AKMU (악뮤),2023.08.21,2023,8,50
116,36416114,Spicy,aespa,2023.05.08,2023,5,4
117,36805679,Vancouver 2,BIG Naughty (서동현),2023.09.19,2023,10,50
118,35008527,아버지,임영웅,2022.05.02,2023,1,50


### 4.3 Release Date Consistency Check

#### Objective
Identify records whose release date is later than their chart month.

#### Procedure
- Parse release dates (`YYYY.MM.DD` and `YYYY.MM`).
- Compare each release date with the last day of its chart month.

#### Results
- Total chart records examined: 25,876
- Records with release dates later than their chart month: 214 (0.83%)
- Several discrepancies were identified in early chart records, including songs appearing in the 2000 charts with release dates recorded in 2001–2003.

#### Key Findings
- Temporal inconsistencies were detected in 0.83% of chart records.
- These discrepancies may reflect incorrect release dates, differences between original and re-released recordings, or mismatched song metadata.
- The results indicate that release dates should not be treated as fully reliable without further validation.
- Additional verification against official Melon records is required to determine the causes of these inconsistencies.

In [12]:

import pandas as pd

validation_df = df.copy()

# Parse release dates (supports YYYY.MM.DD and YYYY.MM)
validation_df["release_date_parsed"] = pd.to_datetime(
    validation_df["release_date"].replace("", pd.NA),
    format="mixed",
    errors="coerce"
)

# Use the last day of each chart month
validation_df["chart_month_end"] = (
    pd.to_datetime(
        validation_df["chart_year"].astype(str) + "-" +
        validation_df["chart_month"].astype(str) + "-01"
    ) + pd.offsets.MonthEnd(0)
)

# Identify songs released after their chart appearance
invalid_dates = validation_df[
    validation_df["release_date_parsed"] >
    validation_df["chart_month_end"]
].copy()

print(f"Total records: {len(validation_df):,}")
print(f"Invalid release dates: {len(invalid_dates):,}")
print(f"Percentage: {len(invalid_dates)/len(validation_df)*100:.2f}%")

display(
    invalid_dates[
        ["song_id", "song_name", "artist",
         "release_date", "chart_year",
         "chart_month", "chart_rank"]
    ].head(20)
)


Total records: 25,876
Invalid release dates: 214
Percentage: 0.83%


,song_id,song_name,artist,release_date,chart_year,chart_month,chart_rank
9,489866,비젼,유승준,2001.09.01,2000,1,10
24,494157,다 알아요,양파,2003.11.13,2000,1,25
30,422691,고요한 밤 거룩한 밤,조성모,2002.01,2000,1,31
35,489869,연가,유승준,2001.09.01,2000,1,36
44,1698079,애수(哀愁),이문세,2002.02.09,2000,1,45
54,1698078,슬픈 사랑의 노래 (WITH 이소라),이문세,2002.02.09,2000,1,50
82,489869,연가,유승준,2001.09.01,2000,2,8
99,494157,다 알아요,양파,2003.11.13,2000,2,25
106,1698078,슬픈 사랑의 노래 (WITH 이소라),이문세,2002.02.09,2000,2,32
152,489869,연가,유승준,2001.09.01,2000,3,16


### 4.4 Temporal Discrepancy Distribution

#### Objective
Examine whether release-date inconsistencies are concentrated in particular chart years.

#### Procedure
- Group records with temporal inconsistencies by chart year.
- Calculate the number and percentage of inconsistent records for each year.
- Compare discrepancy rates across the dataset's coverage period.

#### Results
- The highest discrepancy rate was observed in 2001 (8.97%, 68 records).
- The discrepancy rate in 2000 was 4.66% (36 records).
- A total of 161 discrepancies (75.2% of all 214 cases) occurred between 2000 and 2005.
- Discrepancy rates were generally below 1% from 2009 onward.

#### Key Findings
- Release-date inconsistencies were disproportionately concentrated in earlier chart years.
- The findings suggest that release-date metadata may be less consistent in the early portion of the dataset.
- The causes of these discrepancies remain unverified.
- Further validation is necessary before determining whether affected records should be corrected or excluded.

In [13]:
yearly_discrepancies = (
    invalid_dates.groupby("chart_year")
    .size()
    .reindex(sorted(df["chart_year"].unique()), fill_value=0)
    .rename("invalid_records")
    .to_frame()
)

yearly_discrepancies["total_records"] = (
    df.groupby("chart_year").size()
)

yearly_discrepancies["invalid_percentage"] = (
    yearly_discrepancies["invalid_records"] /
    yearly_discrepancies["total_records"] * 100
).round(2)

display(yearly_discrepancies)


,invalid_records,total_records,invalid_percentage
chart_year,,,
2000,36,773,4.66
2001,68,758,8.97
2002,12,728,1.65
2003,9,755,1.19
2004,19,836,2.27
2005,17,1160,1.47
2006,7,1189,0.59
2007,15,1196,1.25
2008,3,1194,0.25


### 4.5 Flagged Record Sample

#### Objective
Select a sample of flagged songs for verification against official Melon records.

#### Procedure
- Keep one record per song among the release-date discrepancies.
- Randomly sample 10 songs (`random_state=42`).

In [14]:
flagged_sample = (
    invalid_dates
    .dropna(subset=["song_id"])
    .drop_duplicates(subset=["song_id"])
    .sample(
        n=min(10, invalid_dates["song_id"].dropna().nunique()),
        random_state=42
    )
    .sort_values("chart_year")
    .reset_index(drop=True)
)

display(
    flagged_sample[
        ["song_id", "song_name", "artist",
         "release_date", "chart_year",
         "chart_month", "chart_rank"]
    ]
)


,song_id,song_name,artist,release_date,chart_year,chart_month,chart_rank
0,3884179,성녀,이상은 (=Lee-tzsche),2012.08.21,2000,10,37
1,1698079,애수(哀愁),이문세,2002.02.09,2000,1,45
2,489863,찾길 바래,유승준,2001.09.01,2000,12,4
3,28132,연인(戀人),박상민,2001.07.26,2001,6,36
4,530208,시간에 기대어,이브,2004.06.25,2002,12,24
5,720153,믿음,아우라 (Aura),2005.03.02,2005,2,46
6,848146,추억은 시간이 지운다,유미,2005.06.03,2005,5,50
7,1355089,잘해주지 말아요,BGH4,2006.09.08,2006,8,50
8,1574696,하루살이,BMK,2007.03.01,2007,2,50
9,7854179,취향저격,iKON,2015.11.16,2015,9,6


### 4.6 Export Temporal Discrepancy Records

#### Objective
Save the release-date discrepancies for later review.

#### Results
- `data/interim/temporal_discrepancies_214.csv`: all flagged chart records (214)
- `data/interim/temporal_discrepancies_unique_songs.csv`: unique songs with empty verification columns (108)

In [15]:
from pathlib import Path
import pandas as pd

# Output directory
output_dir = Path("../data/interim")
output_dir.mkdir(parents=True, exist_ok=True)

# 1. All flagged chart records (214 records)
columns = [
    "song_id", "song_name", "artist", "album",
    "release_date", "chart_year", "chart_month",
    "chart_rank"
]

all_discrepancies = (
    invalid_dates[columns]
    .sort_values(["chart_year", "chart_month", "chart_rank"])
    .reset_index(drop=True)
)

all_discrepancies.to_csv(
    output_dir / "temporal_discrepancies_214.csv",
    index=False,
    encoding="utf-8-sig"
)

# 2. Unique songs for manual verification
unique_songs = (
    all_discrepancies
    .drop_duplicates(subset=["song_id"])
    .copy()
    .reset_index(drop=True)
)

# Add verification columns
unique_songs["original_release_date"] = ""
unique_songs["discrepancy_type"] = "Unverified"
unique_songs["verification_status"] = "Pending"
unique_songs["evidence_url"] = ""
unique_songs["notes"] = ""
unique_songs["recommended_action"] = ""

unique_songs.to_csv(
    output_dir / "temporal_discrepancies_unique_songs.csv",
    index=False,
    encoding="utf-8-sig"
)

# Summary
print(f"Total flagged records: {len(all_discrepancies):,}")
print(f"Unique songs: {len(unique_songs):,}")
print(f"\nFiles saved to: {output_dir.resolve()}")

display(unique_songs.head(10))


Total flagged records: 214
Unique songs: 108

Files saved to: /Users/ryulha/Desktop/rk/S/Y/2026-2/Music Information Retrieval/kpop-lyrics-nlp/data/interim


,song_id,song_name,artist,album,release_date,chart_year,chart_month,chart_rank,original_release_date,discrepancy_type,verification_status,evidence_url,notes,recommended_action
0,489866,비젼,유승준,You Seung Jun Best & J Duet Collection,2001.09.01,2000,1,10,,Unverified,Pending,,,
1,494157,다 알아요,양파,Yangpa Best,2003.11.13,2000,1,25,,Unverified,Pending,,,
2,422691,고요한 밤 거룩한 밤,조성모,Christmas Memories(Remake Special),2002.01,2000,1,31,,Unverified,Pending,,,
3,489869,연가,유승준,You Seung Jun Best & J Duet Collection,2001.09.01,2000,1,36,,Unverified,Pending,,,
4,1698079,애수(哀愁),이문세,Old And New,2002.02.09,2000,1,45,,Unverified,Pending,,,
5,1698078,슬픈 사랑의 노래 (WITH 이소라),이문세,Old And New,2002.02.09,2000,1,50,,Unverified,Pending,,,
6,4050182,Cross,엄정화,코리안 댄스 탑 40 Vol.1,2000.07.06,2000,4,50,,Unverified,Pending,,,
7,445159,그대가 그대를,이승환,His Ballad Ⅱ,2003.03.12,2000,5,31,,Unverified,Pending,,,
8,3454850,Age Of Peace,H.O.T.,Age Of Peace OST,2010.07.15,2000,8,20,,Unverified,Pending,,,
9,3454851,OP.T (Operation Takeover),H.O.T.,Age Of Peace OST,2010.07.15,2000,8,50,,Unverified,Pending,,,


### 4.7 Monthly Record Counts

#### Objective
Check how many records exist for each chart month.

#### Procedure
- Count records by chart year and month.

#### Results
- 2000–2004 (Jan–Oct): about 53–75 records per month.
- From 2004-11: about 85–100 records per month.
- Missing months (0 records): 2014-06 to 2014-09, 2023-03.
- 2023-11 and 2023-12 are outside the dataset coverage.

#### Key Findings
- The number of records per month increases sharply from November 2004.
- Five chart months are entirely missing from the dataset.

In [16]:
# 4.7 Monthly Record Counts
file_counts = (
    df.groupby(["chart_year", "chart_month"])
      .size()
      .unstack(fill_value=0)
      .reindex(columns=range(1, 13), fill_value=0)
)
file_counts["Total"] = file_counts.sum(axis=1)

display(file_counts)

chart_month,1,2,3,4,5,6,7,8,9,10,11,12,Total
chart_year,,,,,,,,,,,,,
2000,75,62,63,67,69,63,55,69,59,64,66,61,773
2001,64,65,58,65,64,64,63,64,64,58,62,67,758
2002,64,61,56,59,63,58,59,53,66,62,58,69,728
2003,62,66,65,63,66,63,64,59,60,59,63,65,755
2004,59,70,61,67,66,66,70,62,67,65,85,98,836
2005,94,95,97,97,94,97,96,98,98,98,98,98,1160
2006,99,99,98,100,98,98,99,98,100,100,100,100,1189
2007,99,99,100,100,100,100,100,100,99,100,99,100,1196
2008,100,99,99,99,100,100,100,100,100,99,99,99,1194


### 4.8 Manual Verification Checklist

#### Objective
Prepare a checklist for manually comparing top-ranked songs with Melon Chart Finder.

#### Procedure
- Extract ranks 1–3 for every month from 2000 to 2005.
- Add empty columns for the official Melon song, artist, and match result.
- Export to `data/interim/manual_check_top3_2000_2005.csv`.

#### Results
- Checklist rows: 216 / 216 expected
- Missing slots: 0

In [17]:
# 4.8 Manual Verification Checklist: top 3 per month (2000–2005)
checklist = (
    df[(df.chart_year <= 2005) & (df.chart_rank <= 3)]
    .sort_values(["chart_year", "chart_month", "chart_rank"])
    .drop_duplicates(["chart_year", "chart_month", "chart_rank"])
    [["chart_year", "chart_month", "chart_rank", "song_id", "song_name", "artist"]]
    .reset_index(drop=True)
)

# Empty columns to fill in while checking Melon Chart Finder
checklist["melon_song"] = ""
checklist["melon_artist"] = ""
checklist["match"] = ""        # Y / N
checklist["notes"] = ""

# Months where the dataset is missing any of ranks 1–3
expected = pd.MultiIndex.from_product(
    [range(2000, 2006), range(1, 13), range(1, 4)],
    names=["chart_year", "chart_month", "chart_rank"]
)
present = pd.MultiIndex.from_frame(checklist[["chart_year", "chart_month", "chart_rank"]])
missing = expected.difference(present)
print(f"Checklist rows: {len(checklist)} / {len(expected)} expected")
print(f"Missing slots: {len(missing)}")
if len(missing):
    display(missing.to_frame(index=False))

checklist.to_csv(output_dir / "manual_check_top3_2000_2005.csv",
                 index=False, encoding="utf-8-sig")

pd.set_option("display.max_rows", 300)
display(checklist)

Checklist rows: 216 / 216 expected
Missing slots: 0


,chart_year,chart_month,chart_rank,song_id,song_name,artist,melon_song,melon_artist,match,notes
0,2000,1,1,38541,부메랑,김건모,,,,
1,2000,1,2,85257,바꿔,이정현,,,,
2,2000,1,3,91324,사랑해 그리고 기억해,god,,,,
3,2000,2,1,85674,영원,스카이,,,,
4,2000,2,2,607552,가시나무,조성모,,,,
5,2000,2,3,85257,바꿔,이정현,,,,
6,2000,3,1,86668,나의 연인(我戀),임창정,,,,
7,2000,3,2,607552,가시나무,조성모,,,,
8,2000,3,3,1606631,Cyber Lover,터보,,,,
9,2000,4,1,46218,전설속의 누군가처럼,신승훈,,,,


### 4.9 Month-Level Anomaly Detection

#### Objective
Detect chart months whose song lists are inconsistent with neighboring months, without external data.

#### Procedure
- Compute Jaccard overlap of song IDs between each month and its previous/next month.
- Find the most similar month within ±12 months (expected: an adjacent month).
- Measure the share of songs present in both neighboring months but missing in the current month.
- Count songs released after their chart month (from 4.3).
- Flag months that meet any of the following:
  - Overlap ≥ 0.9 with another month (duplicate month)
  - Most similar month is not adjacent
  - Hole ratio ≥ 0.5
  - Late releases ≥ 10
  - Overlap with previous month < 0.2

#### Results
- 2001-06 and 2001-08 have identical song lists (overlap 1.00).
- 2001-06 contains 20 songs released after June 2001.
- 2004-11 shares only 8% of its songs with 2004-10.

#### Key Findings
- 2001-06 contains a copy of the 2001-08 chart; the original June 2001 data is not available in the dataset.
- The discontinuity at 2004-11 coincides with the increase in monthly record counts (4.7) and requires verification against Melon Chart Finder.
- Months following missing months (e.g., 2014-10) may show low overlap because their previous month is not available.

In [18]:
# 4.9 Month-Level Anomaly Detection (no external data needed)
d = df[df.song_id.notna() & df.song_id.str.strip().ne("")].copy()
d["ym"] = d.chart_year * 100 + d.chart_month
months = sorted(d.ym.unique())
sets = d.groupby("ym").song_id.apply(set)

def jaccard(a, b):
    return len(a & b) / len(a | b) if (a | b) else 0.0

# Release-after-chart counts per month (from section 4.3)
late_release = invalid_dates.groupby(["chart_year", "chart_month"]).size()

rows = []
for i, ym in enumerate(months):
    s = sets[ym]
    prev_s = sets[months[i - 1]] if i > 0 else None
    next_s = sets[months[i + 1]] if i < len(months) - 1 else None

    # 1) Overlap with adjacent months
    prev_ov = jaccard(s, prev_s) if prev_s is not None else None
    next_ov = jaccard(s, next_s) if next_s is not None else None

    # 2) Most similar month within ±12 months (should be an adjacent month)
    window = [m for j, m in enumerate(months) if m != ym and abs(j - i) <= 12]
    # Ties are broken in favor of the nearest month
    best = max(window, key=lambda m: (jaccard(s, sets[m]), -abs(months.index(m) - i)))
    best_ov = jaccard(s, sets[best])
    best_dist = abs(months.index(best) - i)

    # 3) "Holes": songs in both prev and next month but missing in this month
    holes = None
    if prev_s is not None and next_s is not None and (prev_s & next_s):
        holes = len((prev_s & next_s) - s) / len(prev_s & next_s)

    y, m = divmod(ym, 100)
    rows.append({
        "year": y, "month": m, "n_songs": len(s),
        "prev_overlap": prev_ov, "next_overlap": next_ov,
        "best_match": f"{best // 100}-{best % 100:02d}",
        "best_overlap": best_ov, "best_distance": best_dist,
        "hole_ratio": holes,
        "late_releases": late_release.get((y, m), 0),
    })

anomaly = pd.DataFrame(rows)

# Flags
anomaly["flag_duplicate_month"] = anomaly.best_overlap >= 0.9          # near-identical to another month
anomaly["flag_nonadjacent_best"] = anomaly.best_distance > 1           # most similar month is not a neighbor
anomaly["flag_holes"] = anomaly.hole_ratio >= 0.5                      # many songs vanish for one month
anomaly["flag_late_release"] = anomaly.late_releases >= 10             # songs charting before release
anomaly["flag_low_continuity"] = anomaly.prev_overlap < 0.2            # little overlap with previous month
flag_cols = [c for c in anomaly.columns if c.startswith("flag_")]
anomaly["n_flags"] = anomaly[flag_cols].sum(axis=1)

flagged = anomaly[anomaly.n_flags > 0].sort_values(["n_flags", "year", "month"],
                                                   ascending=[False, True, True])
print(f"Flagged months: {len(flagged)} / {len(anomaly)}")
display(flagged.round(2))

Flagged months: 4 / 281


,year,month,n_songs,prev_overlap,next_overlap,best_match,best_overlap,best_distance,hole_ratio,late_releases,flag_duplicate_month,flag_nonadjacent_best,flag_holes,flag_late_release,flag_low_continuity,n_flags
17,2001,6,63,0.17,0.60,2001-08,1.00,2,0.41,20,True,True,False,True,True,4
19,2001,8,63,0.60,0.63,2001-06,1.00,2,0.00,5,True,True,False,False,False,2
58,2004,11,78,0.08,0.23,2004-12,0.23,1,0.11,4,False,False,False,False,True,1
173,2014,10,98,0.04,0.39,2014-11,0.39,1,0.00,0,False,False,False,False,True,1


## 5. EDA Summary

| Issue | Records / Months | Section |
|---|---|---|
| Missing song ID, release date, genre | 346 records | 3.2–3.4 |
| Missing release date and genre only | 2 records | 3.4 |
| Same-month duplicates | 6 groups (12 records) | 3.6 |
| Records without valid lyrics | 640 records | 3.7 |
| Release date later than chart month | 214 records (108 songs) | 4.3–4.4 |
| Missing chart months | 2014-06 to 2014-09, 2023-03 | 4.7 |
| Month duplicated from another month | 2001-06 (copy of 2001-08) | 4.9 |
| Discontinuity in chart composition | 2004-11 | 4.7, 4.9 |